# Три сида: отбор чекпоинта по gold-dev

**Зачем.** Одиночный прогон дал gold F1 0.9415 против 0.8680 у базового (отбор по
silver_val, эпоха 3) на одних и тех же 303 диалогах. Нужно заменить одиночное
число на среднее ± std по трём сидам и показать, что расхождение кривых
`silver_val F1` и `gold_dev F1` — устойчивый эффект, а не случайность одного запуска.

**Что нельзя нарушать.**

- Метки train (1392) и val (205) — silver (DeepSeek). Отбор чекпоинта по
  `silver_val` даёт эпоху 3 и качество 0.868 F1 на gold — так делать нельзя.
- Отбор — по **gold-dev** (100 диалогов ручной разметки), финальный замер — на
  оставшихся **303**. Разбиение берётся ГОТОВОЕ из `gold_dev_split.json`
  (seed 20260808, групповое по `scenario_group`, стратификация lang × is_scam,
  пересечение групп между dev и test = 0). Заново не пересоздаётся.
- `gold_test_v1.csv` только читается. Сплиты train/val/test не переразбиваются.

**Конфигурация.** `xlm-roberta-base`, lr 2e-5, weight_decay 0.01, **warmup_ratio 0.2**,
batch 16, max_length 512, 5 эпох, ранняя остановка patience=2 по `eval_gold_dev_f1`.
Меняется только сид: **20260721, 20260722, 20260723**.

⚠ warmup_ratio = 0.2, а не 0.1: именно 0.2 стоял в прогоне, давшем 0.9415, и только
при нём числа сравнимы с базовым результатом.

**Kaggle:** Accelerator = GPU T4, Internet = On, датасет `scam-onset-v1`
(`corpus_v1.jsonl`, `gold_test_v1.csv`, `gold_dev_split.json`). ~5 мин на сид.

## Конфигурация и данные

In [ ]:
import os, sys, json, csv, glob, pathlib, time, copy

def find_one(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"не найден {name} под /kaggle/input — подключён ли датасет?"
    return hits[0]

CORPUS   = find_one("corpus_v1.jsonl")
GOLD     = find_one("gold_test_v1.csv")
GOLD_DEV = find_one("gold_dev_split.json")

OUT = pathlib.Path("/kaggle/working/seeds_gold_dev"); (OUT / "plots").mkdir(parents=True, exist_ok=True)
TB_ROOT = pathlib.Path("/kaggle/working/tb")

MODEL_NAME = "xlm-roberta-base"
SEEDS = [20260721, 20260722, 20260723]
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.2
EPOCHS, MAX_LENGTH, BATCH_SIZE = 5, 512, 16
PATIENCE = 2

import torch, transformers
DEVICE, USE_FP16 = ("cuda", True) if torch.cuda.is_available() else ("cpu", False)
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("устройство:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("GPU в системе:", torch.cuda.device_count() if DEVICE == "cuda" else 0)

In [ ]:
def read_corpus(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: out.append(json.loads(line))
    return out

def build_text(rec):
    return "\n".join(f"[{t['i']}|{t['speaker']}] {t['text']}" for t in rec["turns"])

def read_gold_csv(path):
    out = {}
    with open(path, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            did = (r.get("dialog_id") or "").strip()
            if did: out[did] = int(r["is_scam"])
    return out

records = read_corpus(CORPUS)
gold    = read_gold_csv(GOLD)

train_recs = [r for r in records if r.get("split") == "train"]
val_recs   = [r for r in records if r.get("split") == "val"]

test_split = [r for r in records if r.get("split") == "test"]
all_test = []
for r in test_split:
    r2 = {k: v for k, v in r.items() if k != "is_scam"}    # silver-метку выбрасываем
    if r2["id"] in gold:
        r2["gold_is_scam"] = gold[r2["id"]]
        all_test.append(r2)
assert len(all_test) == len(gold), f"джойн неполный: {len(all_test)} из {len(gold)}"

split_meta = json.load(open(GOLD_DEV, encoding="utf-8"))
assign = split_meta["assignment"]
dev_recs  = [r for r in all_test if assign.get(r["id"]) == "dev"]
test_recs = [r for r in all_test if assign.get(r["id"]) != "dev"]

print(f"разбиение gold: seed={split_meta['seed']}  dev={len(dev_recs)}  test={len(test_recs)}")
assert len(dev_recs) == split_meta["n_dev"] and len(test_recs) == split_meta["n_test"], "разбиение не совпало"
# группы не должны пересекаться между dev и test
gd = {r.get("scenario_group") for r in dev_recs}
gt = {r.get("scenario_group") for r in test_recs}
print(f"групп scenario_group: dev {len(gd)}, test {len(gt)}, пересечение {len(gd & gt)}")
assert not (gd & gt), "группы разорваны между dev и test"
print(f"train {len(train_recs)} (silver) / val {len(val_recs)} (silver) / "
      f"gold-dev {len(dev_recs)} / gold-test {len(test_recs)}")

## Один прогон как функция

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)
from sklearn.metrics import (accuracy_score, cohen_kappa_score, roc_auc_score,
                             average_precision_score, confusion_matrix,
                             precision_recall_fscore_support)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def metrics_from(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                 pos_label=1, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)), "precision": float(p),
            "recall": float(r), "f1": float(f),
            "cohen_kappa": float(cohen_kappa_score(y_true, y_pred))}

class DlgDataset(torch.utils.data.Dataset):
    def __init__(self, recs, label_key):
        self.enc = tok([build_text(r) for r in recs], truncation=True,
                       max_length=MAX_LENGTH, padding="max_length")
        self.labels = [int(r[label_key]) for r in recs]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i]); return item

ds_train_c = DlgDataset(train_recs, "is_scam")
ds_val_c   = DlgDataset(val_recs,   "is_scam")
ds_dev_c   = DlgDataset(dev_recs,   "gold_is_scam")
ds_test_c  = DlgDataset(test_recs,  "gold_is_scam")
print("токенизация готова (одинакова для всех сидов — меняется только инициализация и порядок)")

def run_seed(seed):
    set_seed(seed)
    t0 = time.time()
    tb_dir = TB_ROOT / f"{MODEL_NAME.split('/')[-1]}_seed{seed}_e{EPOCHS}_golddev"
    tb_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TENSORBOARD_LOGGING_DIR"] = str(tb_dir)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        # ФИНАЛЬНАЯ модель: веса обязаны пережить сессию. save_total_limit=2 и
        # обязательный trainer.save_model(...) наружу — см. CLAUDE.md,
        # «Конвенции: сохранение моделей».
    targs = TrainingArguments(
        output_dir=str(OUT / f"ckpt_seed{seed}"),
        learning_rate=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        seed=seed, data_seed=seed, fp16=USE_FP16,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="eval_gold_dev_f1",
        greater_is_better=True, save_total_limit=2, logging_steps=10,
        report_to="tensorboard", logging_dir=str(tb_dir),
    )
    # Пер-диалоговые предсказания на gold-dev по КАЖДОЙ эпохе — нужны для бутстрапа
    # устойчивости отбора без переобучения. Наборы различаются размером, поэтому
    # gold_dev опознаётся по длине; равенство размеров сделало бы это неоднозначным.
    sizes = {len(dev_recs), len(val_recs), len(test_recs)}
    assert len(sizes) == 3, "размеры eval-наборов совпали — опознать набор по длине нельзя"
    dev_epoch_preds, test_epoch_preds = [], []
    capture = {"on": True}          # выключается перед финальным predict

    def compute_metrics(ep):
        logits, labels = ep
        preds = np.argmax(logits, axis=-1)
        if capture["on"] and len(labels) in (len(dev_recs), len(test_recs)):
            probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
            snap = {"y_true": np.asarray(labels).tolist(), "y_pred": preds.tolist(),
                    "prob_scam": probs.tolist()}
            (dev_epoch_preds if len(labels) == len(dev_recs) else test_epoch_preds).append(snap)
        return metrics_from(labels, preds)

    trainer = Trainer(model=model, args=targs, train_dataset=ds_train_c,
                      eval_dataset={"silver_val": ds_val_c, "gold_dev": ds_dev_c,
                                    "gold_test": ds_test_c},
                      processing_class=tok, compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()
    hist = copy.deepcopy(trainer.state.log_history)

    # ── ход по эпохам: train loss + обе метрики ──
    tr = [h for h in hist if "loss" in h and "eval_silver_val_loss" not in h and "eval_gold_dev_loss" not in h]
    sv = [h for h in hist if "eval_silver_val_f1" in h]
    gv = [h for h in hist if "eval_gold_dev_f1" in h]
    per_epoch = []
    for a, b in zip(sv, gv):
        ep = a["epoch"]
        v = [x["loss"] for x in tr if ep - 1 < x["epoch"] <= ep]
        rec = {"epoch": int(round(ep)), "seed": seed,
               "train_loss": float(sum(v) / len(v)) if v else None,
               "silver_val_f1": float(a["eval_silver_val_f1"]),
               "gold_dev_f1": float(b["eval_gold_dev_f1"])}
        gt = next((h for h in hist if "eval_gold_test_f1" in h and h["epoch"] == ep), None)
        if gt:
            for k in ("f1", "precision", "recall", "accuracy", "cohen_kappa"):
                rec[f"gold_test_{k}"] = float(gt[f"eval_gold_test_{k}"])
        per_epoch.append(rec)

    best_epoch = max(per_epoch, key=lambda x: x["gold_dev_f1"])["epoch"]

    # ── тест лучшего чекпоинта на 303 ──
    capture["on"] = False        # финальный predict не должен попасть в снимки
    po = trainer.predict(ds_test_c)
    logits = po.predictions
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    y_pred = logits.argmax(axis=-1)
    y_true = np.array([r["gold_is_scam"] for r in test_recs])

    m = metrics_from(y_true, y_pred)
    m["roc_auc"] = float(roc_auc_score(y_true, probs))
    m["average_precision"] = float(average_precision_score(y_true, probs))
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    per_lang = {}
    for lg in sorted({r.get("lang") for r in test_recs if r.get("lang")}):
        idx = [i for i, r in enumerate(test_recs) if r.get("lang") == lg]
        per_lang[lg] = metrics_from(y_true[idx], y_pred[idx]) | {"n": len(idx)}

    silver = {r["id"]: int(r["is_scam"]) for r in records}
    same_silver = sum(1 for r, p in zip(test_recs, y_pred) if silver[r["id"]] == int(p))

    from torch.utils.tensorboard import SummaryWriter
    w = SummaryWriter(log_dir=str(tb_dir))
    for k, v in m.items(): w.add_scalar(f"gold_test/{k}", v, 0)
    for lg, mm in per_lang.items(): w.add_scalar(f"gold_test_f1/{lg}", mm["f1"], 0)
    w.flush(); w.close()

    res = {"seed": seed, "per_epoch": per_epoch, "best_epoch": best_epoch,
           "epochs_ran": len(per_epoch),
           "best_checkpoint": trainer.state.best_model_checkpoint,
           "best_gold_dev_f1": float(trainer.state.best_metric),
           "test_metrics": m, "confusion_matrix": cm.tolist(),
           "per_language": per_lang, "n_test": len(test_recs),
           "silver_agreement": same_silver / len(test_recs),
           "minutes": round((time.time() - t0) / 60, 1),
           "probs": probs.tolist(), "y_pred": y_pred.tolist()}
    json.dump(hist, open(OUT / f"history_seed{seed}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    # эпоха i+1 ↔ dev_epoch_preds[i]; порядок диалогов = порядок dev_recs
    assert len(dev_epoch_preds) == len(per_epoch),         f"эпох {len(per_epoch)}, а снимков gold-dev {len(dev_epoch_preds)}"
    assert len(test_epoch_preds) == len(per_epoch),         f"эпох {len(per_epoch)}, а снимков gold-test {len(test_epoch_preds)}"
    json.dump({"seed": seed, "dialog_ids": [r["id"] for r in test_recs],
               "langs": [r.get("lang") for r in test_recs],
               "epochs": [p["epoch"] for p in per_epoch],
               "per_epoch": test_epoch_preds},
              open(OUT / f"gold_test_per_epoch_seed{seed}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)
    json.dump({"seed": seed, "dialog_ids": [r["id"] for r in dev_recs],
               "langs": [r.get("lang") for r in dev_recs],
               "epochs": [p["epoch"] for p in per_epoch],
               "per_epoch": dev_epoch_preds},
              open(OUT / f"gold_dev_per_epoch_seed{seed}.json", "w", encoding="utf-8"),
              ensure_ascii=False, indent=2)

    del trainer, model
    torch.cuda.empty_cache()
    print(f"\n[seed {seed}] {res['minutes']} мин | эпох {res['epochs_ran']} | "
          f"лучшая эпоха {best_epoch} | gold F1 {m['f1']:.4f}")
    return res

## Прогоны

In [ ]:
ALL = []
for s in SEEDS:
    print("\n" + "=" * 74)
    print(f"СИД {s}")
    print("=" * 74)
    ALL.append(run_seed(s))

## Ход по эпохам: обе метрики, три сида

In [ ]:
print(f"{'сид':>10} {'эпоха':>6} {'train loss':>11} {'silver_val F1':>14} {'gold_dev F1':>12}  {'':>3}")
print("-" * 62)
for res in ALL:
    for r in res["per_epoch"]:
        mark = " ←" if r["epoch"] == res["best_epoch"] else ""
        tl = f"{r['train_loss']:.4f}" if r["train_loss"] is not None else "  —   "
        print(f"{r['seed']:>10} {r['epoch']:>6} {tl:>11} {r['silver_val_f1']:>14.4f} "
              f"{r['gold_dev_f1']:>12.4f}{mark}")
    print("-" * 62)
print("← лучшая эпоха по gold_dev F1 (по ней взят чекпоинт)")

## Результаты по сидам и сводка

In [ ]:
KEYS = ["accuracy", "precision", "recall", "f1", "cohen_kappa", "roc_auc", "average_precision"]

print(f"gold-test, n={ALL[0]['n_test']}\n")
hdr = f"{'сид':>10} {'эпоха':>6} " + " ".join(f"{k[:9]:>10}" for k in KEYS)
print(hdr); print("-" * len(hdr))
for res in ALL:
    print(f"{res['seed']:>10} {res['best_epoch']:>6} " +
          " ".join(f"{res['test_metrics'][k]:>10.4f}" for k in KEYS))

summary = {}
print("\n" + "=" * 46)
print(f"{'метрика':>20} {'среднее':>10} {'± std':>9}")
print("=" * 46)
for k in KEYS:
    vals = [r["test_metrics"][k] for r in ALL]
    mu, sd = float(np.mean(vals)), float(np.std(vals, ddof=1))
    summary[k] = {"mean": mu, "std": sd, "values": vals}
    print(f"{k:>20} {mu:>10.4f} {sd:>9.4f}")
sa = [r["silver_agreement"] for r in ALL]
summary["silver_agreement"] = {"mean": float(np.mean(sa)), "std": float(np.std(sa, ddof=1)), "values": sa}
print(f"{'совпадение с silver':>20} {np.mean(sa):>10.4f} {np.std(sa, ddof=1):>9.4f}")

best_eps = [r["best_epoch"] for r in ALL]
ran = [r["epochs_ran"] for r in ALL]
print(f"\nлучшая эпоха по сидам: {best_eps}   (размах {max(best_eps) - min(best_eps)})")
print(f"эпох отработано:       {ran}  из {EPOCHS} — ранняя остановка "
      f"{'срабатывала' if any(x < EPOCHS for x in ran) else 'не срабатывала'}")

print("\nпо языкам (среднее ± std):")
for lg in sorted(ALL[0]["per_language"]):
    for met in ("f1", "precision", "recall"):
        v = [r["per_language"][lg][met] for r in ALL]
        print(f"  {lg} {met:>9}: {np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}", end="")
    print(f"   (n={ALL[0]['per_language'][lg]['n']})")

cm_mean = np.mean([np.array(r["confusion_matrix"]) for r in ALL], axis=0)
print("\nусреднённая матрица ошибок (строки gold, столбцы предсказание):")
print(f"            pred=0    pred=1")
print(f"  gold=0  {cm_mean[0,0]:8.1f}  {cm_mean[0,1]:8.1f}")
print(f"  gold=1  {cm_mean[1,0]:8.1f}  {cm_mean[1,1]:8.1f}")

results = {"seeds": SEEDS, "config": {"model": MODEL_NAME, "lr": LR,
           "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
           "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_length": MAX_LENGTH,
           "patience": PATIENCE, "selection_metric": "eval_gold_dev_f1",
           "device": DEVICE, "n_gpu": torch.cuda.device_count() if DEVICE == "cuda" else 0,
           "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                        "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
                        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
                        "deterministic": bool(torch.are_deterministic_algorithms_enabled()),
                        "cublas_workspace": os.environ.get("CUBLAS_WORKSPACE_CONFIG")}},
           "split": {"file": "gold_dev_split.json", "seed": split_meta["seed"],
                     "n_dev": len(dev_recs), "n_test": len(test_recs)},
           "runs": [{k: v for k, v in r.items() if k != "probs"} for r in ALL],
           "summary": summary, "best_epochs": best_eps, "epochs_ran": ran,
           "confusion_matrix_mean": cm_mean.tolist()}
json.dump(results, open(OUT / "results.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\nсохранено:", OUT / "results.json")

## Графики

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")
PLOTS = OUT / "plots"
COLORS = ["#4C72B0", "#DD8452", "#55A868"]

# divergence.png — главный график
fig, ax = plt.subplots(figsize=(10, 6))
for res, c in zip(ALL, COLORS):
    eps = [r["epoch"] for r in res["per_epoch"]]
    ax.plot(eps, [r["silver_val_f1"] for r in res["per_epoch"]], marker="o", color=c,
            linewidth=2.2, linestyle="-", label=f"silver_val, сид {res['seed']}")
    ax.plot(eps, [r["gold_dev_f1"] for r in res["per_epoch"]], marker="s", color=c,
            linewidth=2.2, linestyle="--", label=f"gold_dev, сид {res['seed']}")
    ax.scatter([res["best_epoch"]],
               [next(r["gold_dev_f1"] for r in res["per_epoch"] if r["epoch"] == res["best_epoch"])],
               s=190, facecolors="none", edgecolors=c, linewidths=2.4, zorder=5)
ax.set_xlabel("эпоха"); ax.set_ylabel("F1")
ax.set_xticks(sorted({r["epoch"] for res in ALL for r in res["per_epoch"]}))
ax.set_title("Расхождение метрик: silver_val растёт, gold_dev падает\n"
             "сплошные — silver (метки DeepSeek), пунктир — gold (ручная разметка), "
             "кружок — выбранный чекпоинт")
ax.legend(fontsize=9, ncol=3, loc="lower center")
fig.tight_layout(); fig.savefig(PLOTS / "divergence.png", dpi=150); plt.show()

# seed_variance.png
fig, ax = plt.subplots(figsize=(10, 6))
f1s = [r["test_metrics"]["f1"] for r in ALL]
mu, sd = float(np.mean(f1s)), float(np.std(f1s, ddof=1))
ax.axhspan(mu - sd, mu + sd, color="#4C72B0", alpha=0.16, label=f"среднее ± std = {mu:.4f} ± {sd:.4f}")
ax.axhline(mu, color="#4C72B0", linewidth=2)
for i, (res, c) in enumerate(zip(ALL, COLORS)):
    ax.scatter([i], [res["test_metrics"]["f1"]], s=150, color=c, zorder=5)
    ax.annotate(f"{res['test_metrics']['f1']:.4f}\n(эпоха {res['best_epoch']})",
                (i, res["test_metrics"]["f1"]), textcoords="offset points",
                xytext=(14, -6), fontsize=10)
ax.set_xticks(range(len(ALL)), [str(r["seed"]) for r in ALL])
ax.set_xlim(-0.5, len(ALL) - 0.2)
ax.set_xlabel("сид"); ax.set_ylabel("F1 на gold-test")
ax.set_title(f"Разброс F1 между сидами на gold-test (n={ALL[0]['n_test']})")
ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(PLOTS / "seed_variance.png", dpi=150); plt.show()

# confusion_mean.png
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(cm_mean, cmap="Blues")
rs = cm_mean.sum(axis=1, keepdims=True)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_mean[i,j]:.1f}\n{cm_mean[i,j]/rs[i,0]:.1%} строки",
                ha="center", va="center", fontsize=16,
                color="white" if cm_mean[i,j] > cm_mean.max()/2 else "#222")
ax.set_xticks([0,1], ["benign (0)", "scam (1)"]); ax.set_yticks([0,1], ["benign (0)", "scam (1)"])
ax.set_xlabel("предсказание"); ax.set_ylabel("gold")
ax.set_title(f"Матрица ошибок на gold-test, усреднённая по {len(ALL)} сидам (n={ALL[0]['n_test']})")
ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8, label="диалогов (среднее)")
fig.tight_layout(); fig.savefig(PLOTS / "confusion_mean.png", dpi=150); plt.show()

print("графики:", sorted(p.name for p in PLOTS.glob("*.png")))